# Silverwing-ML - Production Training on Colab (v2, Kaggle-migrated)

Continues the production run from the Kaggle checkpoints. Same model, data,
hyperparams - checkpoints are interchangeable between platforms.

**One-time setup (browser):**
1. `Runtime` > `Change runtime type` > **T4 GPU**.
2. Click the key icon (Secrets) > add:
   - `KAGGLE_JSON` = full contents of your kaggle.json (`{"username":..., "key":...}`)
   - `WANDB_API_KEY` (optional) = your wandb key -> enables live cloud logging
     that can be monitored remotely.

**Every session:** run cells 1-7 in order, then Cell 8 (pretrain) until COMPLETE,
then 9-13. Cell 8 auto-resumes from Drive; just re-run it if anything dies.

In [ ]:
# Cell 1: Mount Google Drive (all progress persists here)
import os

from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'
for sub in ['checkpoints/pretrain', 'checkpoints/sft', 'corpus/corpus-external', 'tokenizer']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
print('Drive ready:', DRIVE)

### 🔑 Secret Verification
Run this cell to ensure your Kaggle credentials are correctly loaded from the Colab Secrets manager.

In [ ]:
from google.colab import userdata
try:
    user = userdata.get('KAGGLE_USERNAME')
    key = userdata.get('KAGGLE_KEY')
    print(f"✅ Success: Found Kaggle credentials for user: {user}")
except userdata.SecretNotFoundError:
    print("❌ Error: KAGGLE_USERNAME or KAGGLE_KEY not found in Secrets.")
except userdata.NotebookAccessError:
    print("❌ Error: Please enable 'Notebook access' for your Kaggle secrets.")

In [4]:
# Cell 2: ONE-TIME SEED from Kaggle (Using Colab Secrets)
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

try:
    KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
    KAGGLE_KEY = userdata.get('KAGGLE_KEY')
except Exception:
    # Fallback to hardcoded only if secrets aren't set, but secrets are preferred
    KAGGLE_USERNAME = 'videlisndichi'
    KAGGLE_KEY = 'KGAT_4252a1b11f1dc8ea8a0d48a8bee2986d'

STATE_DS = f'videlisndichi/silverwing-state'

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

kg = Path.home() / '.kaggle'
kg.mkdir(exist_ok=True)
(kg / 'kaggle.json').write_text('{"username": "%s", "key": "%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
(kg / 'kaggle.json').chmod(0o600)

if subprocess.run(['kaggle', '--version'], capture_output=True).returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

def _kaggle(*args):
    return subprocess.run(['kaggle', *args], capture_output=True, text=True)

dl = Path('/content/statedl')
shutil.rmtree(dl, ignore_errors=True)
dl.mkdir(parents=True)

print(f"Attempting to download state from {STATE_DS}...")
r = _kaggle('datasets', 'download', STATE_DS, '-p', str(dl), '--unzip')

if r.returncode != 0:
    print(f"Error: {r.stdout}\n{r.stderr}")
    print("Please check if the dataset is private or if your Kaggle API key is correct.")
else:
    state = dl / 'state' if (dl / 'state').exists() else dl

    def latest_step(d: Path) -> int:
        return max([int(m.group(1)) for p in d.glob('step-*.pt')
                    if (m := re.match(r'step-(\d+)\.pt', p.name))] or [0])

    dst_ck = Path(DRIVE) / 'checkpoints/pretrain'
    src_ck = state / 'checkpoints/pretrain'
    if src_ck.exists():
        for f in src_ck.iterdir():
            if f.is_file(): shutil.copy2(f, dst_ck / f.name)

    dst_corpus = Path(DRIVE) / 'corpus/corpus-external'
    src_corpus = state / 'corpus/corpus-external'
    if src_corpus.exists():
        for f in src_corpus.iterdir():
            if f.is_file(): shutil.copy2(f, dst_corpus / f.name)

    print(f'Sync complete. Drive usage: {sum(f.stat().st_size for f in Path(DRIVE).rglob("*") if f.is_file()) / 1e9:.2f} GB')

Attempting to download state from videlisndichi/silverwing-state...
Error: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata


Please check if the dataset is private or if your Kaggle API key is correct.


In [5]:
# Cell 3: Dependencies + compute probe (fails fast on unsupported GPUs)
import sys

import warnings

warnings.filterwarnings('ignore', message='.*CUDA capability.*')
warnings.filterwarnings('ignore', message='.*Found GPU.*')
warnings.filterwarnings('ignore', message='.*Please install PyTorch.*')

!pip install -q pyyaml numpy datasets huggingface_hub zstandard

import datasets
import torch
import zstandard

print(f'torch {torch.__version__} | datasets {datasets.__version__} | '
      f'zstandard {zstandard.__version__}')
if not torch.cuda.is_available():
    sys.exit('NO GPU - Runtime > Change runtime type > T4 GPU, then rerun')
arch_list = torch.cuda.get_arch_list()
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cap = f'sm_{p.major}{p.minor}'
    ok = cap in arch_list or f'compute_{p.major}{p.minor}' in arch_list
    status = 'OK' if ok else 'UNSUPPORTED BY THIS TORCH BUILD'
    print(f'GPU {i}: {p.name} ({p.total_memory / 1e9:.0f} GB, {cap}) -> {status}')
    if not ok:
        sys.exit(f'{p.name} ({cap}) cannot run this torch build.')

torch 2.11.0+cu128 | datasets 4.0.0 | zstandard 0.25.0
GPU 0: Tesla T4 (16 GB, sm_75) -> OK


In [6]:
# Cell 4: Clone the repo and HARD-SYNC to latest origin/main
import os

REPO = '/content/Silverwing-ML'
if not os.path.exists(REPO):
    !git clone https://github.com/oledesug-source/silverwing-ml.git {REPO}
os.chdir(REPO)
!git fetch origin
!git reset --hard origin/main
!git clean -fdq
head = !git rev-parse --short HEAD
print('repo at commit:', head[0])

Cloning into '/content/Silverwing-ML'...
remote: Enumerating objects: 1604, done.
remote: Counting objects: 100% (1604/1604), done.
remote: Compressing objects: 100% (1187/1187), done.
remote: Total 1604 (delta 487), reused 1453 (delta 336), pack-reused 0 (from 0)
Receiving objects: 100% (1604/1604), 14.88 MiB | 13.04 MiB/s, done.
Resolving deltas: 100% (487/487), done.
HEAD is now at 12ef086 fix: pipeline wrote amp=True for --cpu runs (SftConfig rejects amp on cpu); amp now conditional on device
repo at commit: 12ef086


In [7]:
# Cell 5: Corpus from Drive
import shutil
from pathlib import Path

CORPUS = Path('experiments/corpus-external')
S_CORPUS = Path(DRIVE) / 'corpus/corpus-external'

# Ensure the source directory exists to avoid errors if seed failed
S_CORPUS.mkdir(parents=True, exist_ok=True)

if not any(S_CORPUS.glob('train.*.jsonl')):
    print("WARNING: No corpus files found on Drive. Pretraining might fail unless Cell 6b fetches caches.")
else:
    if CORPUS.exists():
        shutil.rmtree(CORPUS)
    shutil.copytree(S_CORPUS, CORPUS, dirs_exist_ok=True)
    train_shard = next(CORPUS.glob('train.*.jsonl'))
    print(f"Local corpus ready. Documents in first shard: {sum(1 for _ in open(train_shard, encoding='utf-8')):,}")

In [8]:
# Cell 6: Tokenizer v2 from Drive
import shutil
from pathlib import Path

TOK = Path('experiments/tokenizer-v2')
D_TOK = Path(DRIVE) / 'tokenizer/tokenizer-v2'
assert (D_TOK / 'vocab.json').exists(), 'no tokenizer on Drive - run Cell 2 (seed) first'
if TOK.exists():
    shutil.rmtree(TOK)
shutil.copytree(D_TOK, TOK)
TOKENIZER_DIR = 'experiments/tokenizer-v2'
print('TOKENIZER_DIR =', TOKENIZER_DIR)

AssertionError: no tokenizer on Drive - run Cell 2 (seed) first

In [ ]:
# Cell 6b: Fetch token caches (skips the ~100 min tokenization tax)
import subprocess

r = subprocess.run(['kaggle', 'datasets', 'download',
                    'videlisndichi/silverwing-tokcache',
                    '-p', 'experiments/corpus-external', '--unzip'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('token caches fetched - tokenization will be skipped')
else:
    print('[warn] tokcache unavailable - first startup will tokenize once '
          '(~100 min), then cache for this session')

In [ ]:
# Cell 7: Production pretraining config - MUST MATCH the Kaggle run exactly
# (the trainer's resume guard rejects config drift; only checkpoint_dir differs)
import yaml

cfg = {'training': {
    'version': 'training-v1',
    'model_config_path': 'configs/model.yaml',
    'corpus_dir': 'experiments/corpus-external',
    'tokenizer_dir': TOKENIZER_DIR,
    'checkpoint_dir': f'{DRIVE}/checkpoints/pretrain',
    'batch_size': 16,
    'grad_accum_steps': 1,
    'block_size': 512,
    'max_steps': 36000,
    'warmup_steps': 2000,
    'lr': 3.0e-4,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.1,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 50,
    'eval_steps': 1000,
    'eval_sequences': 16,
    'save_steps': 1000,
    'verify_dataset': False,
    'expected_dataset_hash': None,
    'require_validation': True,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/training_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
t = cfg['training']
print('configs/training_production.yaml written')
print(f"target: {t['max_steps']:,} steps x {t['batch_size'] * t['block_size']:,} tok/step")

In [ ]:
# Cell 8: PRETRAIN RUNNER - safe to re-run any number of times.
# Before deciding anything, probes the Kaggle state dataset (cheap ~3 KB
# report fetch) and pulls its newer checkpoint files onto Drive if Kaggle
# is ahead - so a stale Drive seed never causes a pointless resume.
# Then: resumes from the newest Drive checkpoint, prunes old ones to
# protect Drive quota, streams live progress with a watchdog heartbeat.
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

STATE_DS = 'videlisndichi/silverwing-state'
CKPT = Path(DRIVE) / 'checkpoints/pretrain'
TARGET = 36000
KEEP_STEPS = 2

def _kaggle(*args):
    return subprocess.run(['kaggle', *args], capture_output=True, text=True)

def latest_step(d: Path) -> int:
    return max([int(m.group(1)) for p in d.glob('step-*.pt')
                if (m := re.match(r'step-(\d+)\.pt', p.name))] or [0])

def prune(old_keep=KEEP_STEPS):
    steps = sorted(CKPT.glob('step-*.pt'),
                   key=lambda p: int(p.stem.split('-')[1]))
    for p in steps[:-old_keep]:
        p.unlink()

def sync_from_kaggle():
    tmp = Path('/content/statepull')
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    r = _kaggle('datasets', 'download', STATE_DS,
                '-f', 'checkpoints/pretrain/training_report.json',
                '-p', str(tmp), '--unzip')
    rep = tmp / 'training_report.json'
    if r.returncode != 0 or not rep.exists():
        print('[sync] kaggle state unreachable - continuing with Drive only')
        shutil.rmtree(tmp, ignore_errors=True)
        return
    remote = int(json.loads(rep.read_text(encoding='utf-8')).get('steps_done') or 0)
    local = latest_step(CKPT)
    if remote <= local:
        print(f'[sync] Drive already current (local {local:,} >= remote {remote:,})')
        shutil.rmtree(tmp, ignore_errors=True)
        return
    print(f'[sync] Kaggle is ahead: {local:,} -> {remote:,}; pulling files...')
    wanted = [f'checkpoints/pretrain/step-{remote:08d}.pt',
              'checkpoints/pretrain/best.pt',
              'checkpoints/pretrain/final.pt']
    for f in wanted:
        r = _kaggle('datasets', 'download', STATE_DS, '-f', f,
                    '-p', str(tmp), '--unzip')
        src = tmp / Path(f).name
        if r.returncode == 0 and src.exists():
            shutil.copy2(src, CKPT / src.name)
        else:
            print(f'[sync] skip {Path(f).name} (fetch failed)')
    shutil.copy2(rep, CKPT / 'training_report.json')
    shutil.rmtree(tmp, ignore_errors=True)
    print(f'[sync] Drive now at step {latest_step(CKPT):,}')

sync_from_kaggle()

done = latest_step(CKPT)
if done >= TARGET and (CKPT / 'best.pt').exists():
    print(f'PRETRAIN COMPLETE ({done:,} steps). Continue to Cell 9.')
else:
    cks = sorted(CKPT.glob('step-*.pt'),
                 key=lambda p: int(re.sub(r'\D', '', p.name)))
    cmd = [sys.executable, 'scripts/train.py',
           '--config', 'configs/training_production.yaml',
           '--device', 'cuda',
           '--no-clean-repo-check']
    if cks:
        cmd += ['--resume-from', str(cks[-1])]
        print(f'Resuming from step {done:,} / {TARGET:,}')
    else:
        print('Fresh pretraining start')
    t0 = time.time()
    proc = subprocess.Popen(cmd, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    polls = 0
    try:
        while proc.poll() is None:
            time.sleep(30)
            polls += 1
            cur = latest_step(CKPT)
            if polls % 10 == 0:
                try:
                    free = shutil.disk_usage(DRIVE).free / 1e9
                    extra = f', Drive free {free:.1f} GB'
                except Exception:
                    extra = ''
                print(f'[watchdog] alive {int((time.time() - t0) / 60)} min, '
                      f'step {cur:,}{extra}', flush=True)
            if cur > 0:
                prune()
    finally:
        rc = proc.wait()
        prune()
        cur = latest_step(CKPT)
        print(f'exit={rc} elapsed={(time.time() - t0) / 60:.1f} min')
        print(f'progress: {cur:,} / {TARGET:,} steps')
        if cur < TARGET:
            print('Run this cell again to continue.')

In [ ]:
# Cell 9: Verify SFT v3 dataset (math + reasoning + general chat, committed in repo)
from pathlib import Path

sft_ds = Path('experiments/sft/sft-v3-all.jsonl')
assert sft_ds.exists(), 'SFT dataset missing from repo - check git clone'
n = sum(1 for _ in open(sft_ds, encoding='utf-8'))
print(f'SFT examples: {n:,}')

In [ ]:
# Cell 10: SFT run (fp16 AMP, init from pretrained best; skips until pretrain done)
# Auto-prunes old SFT step checkpoints while running (keeps best.pt + newest)
# so a 15 GB free Drive account can never fill up mid-run.
import os
import re
import subprocess
import sys
import time

from pathlib import Path

import yaml

PRETRAIN_DONE = max([int(m.group(1)) for p in Path(DRIVE + '/checkpoints/pretrain').glob('step-*.pt')
                     if (m := re.match(r'step-(\d+)\.pt', p.name))] or [0])
PRETRAIN_BEST = f'{DRIVE}/checkpoints/pretrain/best.pt'
print(f'pretrain progress: {PRETRAIN_DONE:,} / 36,000 steps')
if PRETRAIN_DONE >= 36000 and os.path.exists(PRETRAIN_BEST):
    cfg = {'sft': {
        'version': 'sft-v3',
        'model_config_path': 'configs/model.yaml',
        'tokenizer_dir': TOKENIZER_DIR,
        'init_from': PRETRAIN_BEST,
        'dataset_path': 'experiments/sft/sft-v3-all.jsonl',
        'checkpoint_dir': f'{DRIVE}/checkpoints/sft',
        'batch_size': 16,
        'block_size': 512,
        'max_steps': 1200,  # 1200 not 2000: v2 overfit (train_loss 0.002)
        'warmup_steps': 100,
        'lr': 1.0e-4,
        'min_lr_ratio': 0.1,
        'weight_decay': 0.1,
        'betas': [0.9, 0.95],
        'eps': 1.0e-8,
        'grad_clip': 1.0,
        'seed': 42,
        'log_steps': 25,
        'eval_steps': 200,
        'eval_examples': 64,
        'save_steps': 500,
        'eval_fraction': 0.05,
        'require_clean_repo': False,
        'device': 'cpu',
        'amp': True,
        'amp_dtype': 'float16',
    }}
    with open('configs/sft_production.yaml', 'w') as f:
        yaml.safe_dump(cfg, f)
    SFT_DIR = Path(DRIVE) / 'checkpoints/sft'

    def sft_prune(keep=1):
        steps = sorted(SFT_DIR.glob('step-*.pt'),
                       key=lambda p: int(re.sub(r'\D', '', p.name)))
        for p in steps[:-keep]:
            p.unlink()

    cmd = [sys.executable, 'scripts/train_sft.py',
           '--config', 'configs/sft_production.yaml',
           '--device', 'cuda',
           '--no-clean-repo-check']
    t0 = time.time()
    proc = subprocess.Popen(cmd, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    try:
        while proc.poll() is None:
            time.sleep(30)
            sft_prune()
    finally:
        rc = proc.wait()
        sft_prune()
        print(f'SFT exit={rc} elapsed={(time.time() - t0) / 60:.1f} min')
        assert rc == 0, 'SFT run failed'
else:
    print('Skipping SFT: pretrain not finished. Rerun Cell 8, then this cell.')

In [ ]:
# Cell 11: Generation test (skips itself until SFT has run)
import os

import sys

import torch

if not os.path.exists(f'{DRIVE}/checkpoints/sft/best.pt'):
    print('SFT best.pt not ready - skipping generation test. Run Cell 10 after pretraining completes.')
else:
    sys.path.insert(0, '.')
    from foundation.inference import Generator
    from foundation.model.config import ModelConfig
    from foundation.model.model import SilverwingDecoder
    from foundation.tokenizer import TokenizerV2

    tok = TokenizerV2.load(TOKENIZER_DIR)
    ckpt_path = f'{DRIVE}/checkpoints/sft/best.pt'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_cfg = ModelConfig.from_yaml('configs/model.yaml')
    model = SilverwingDecoder(model_cfg)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state', state))
    model = model.to(device).eval()
    print(f'Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params from {ckpt_path}')

    gen = Generator(model, tok)
    prompts = [
        'What is 2 + 2? ',
        'The answer to 3 * 3 is ',
        'Solve for x: 2x + 5 = 13. Step 1:',
        'The capital of France is ',
    ]
    for p in prompts:
        result = gen.generate(p, max_new_tokens=80, temperature=0.7, top_k=50)
        print(f'--- {p!r}\n{result}')

In [ ]:
# Cell 12: Math benchmark against regression gates (skips until SFT has run)
import os

import sys
import subprocess

if os.path.exists(f'{DRIVE}/checkpoints/sft/best.pt'):
    subprocess.run([sys.executable, 'scripts/run_benchmark.py',
                    '--benchmark', 'math-benchmark-v1',
                    '--model', f'silverwing:{DRIVE}/checkpoints/sft/best.pt',
                    '--tokenizer-dir', TOKENIZER_DIR])
else:
    print('SFT best.pt not ready - skipping benchmark. Run Cell 10 after pretraining completes.')

In [ ]:
# Cell 13: Manufacturing summary
import json
from pathlib import Path

for name, report in [('pretrain', Path(DRIVE) / 'checkpoints/pretrain/training_report.json'),
                     ('sft', Path(DRIVE) / 'checkpoints/sft/sft_report.json')]:
    if report.exists():
        d = json.loads(report.read_text(encoding='utf-8'))
        print(f'[{name}] eval_loss={d.get("final_eval_loss")} ppl={d.get("final_perplexity")} '
              f'best={d.get("best_eval_loss")}')
    else:
        print(f'[{name}] report not found yet')
print('\nArtifacts on Drive:')
for p in sorted(Path(DRIVE).rglob('*.pt')):
    print(f'  {p.relative_to(DRIVE)} ({p.stat().st_size / 1e9:.2f} GB)')

In [ ]:
# Cell 11b: DPO alignment run (M17) - init from SFT best, mixed math+general pairs
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import yaml

SFT_BEST = f'{DRIVE}/checkpoints/sft/best.pt'
assert os.path.exists(SFT_BEST), 'SFT best.pt missing - finish the SFT cell first'
assert Path('experiments/alignment/dpo-v2-all.jsonl').exists(), \
    'dpo-v2-all.jsonl missing from repo - pull latest'

cfg = {'alignment': {
    'version': 'alignment-v2',
    'model_config_path': 'configs/model.yaml',
    'tokenizer_dir': TOKENIZER_DIR,
    'init_from': SFT_BEST,
    'dataset_path': 'experiments/alignment/dpo-v2-all.jsonl',
    'checkpoint_dir': f'{DRIVE}/checkpoints/alignment',
    'batch_size': 4,
    'block_size': 512,
    'max_steps': 800,
    'warmup_steps': 40,
    'lr': 1.0e-5,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.0,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'dpo_beta': 0.1,
    'label_smoothing': 0.0,
    'seed': 42,
    'log_steps': 10,
    'eval_steps': 100,
    'eval_examples': 16,
    'save_steps': 200,
    'eval_fraction': 0.05,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/alignment_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

ALIGN_DIR = Path(DRIVE) / 'checkpoints/alignment'

def align_prune(keep=1):
    steps = sorted(ALIGN_DIR.glob('step-*.pt'),
                   key=lambda p: int(re.sub(r'\D', '', p.name)))
    for p in steps[:-keep]:
        p.unlink()

cmd = [sys.executable, 'scripts/train_alignment.py',
       '--config', 'configs/alignment_production.yaml',
       '--device', 'cuda',
       '--no-clean-repo-check']
t0 = time.time()
proc = subprocess.Popen(cmd, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
try:
    while proc.poll() is None:
        time.sleep(30)
        align_prune()
finally:
    rc = proc.wait()
    align_prune()
    print(f'DPO exit={rc} elapsed={(time.time() - t0) / 60:.1f} min')
    if rc == 0:
        report = Path('experiments/alignment') / 'alignment_report.json'
        if report.exists():
            print(json.dumps(json.loads(report.read_text()), indent=2)[:800])
